In [1]:
from scheduling_dpmsolver_multistep import DPMSolverMultistepScheduler
import torch
import torchvision.transforms as transforms
from src.stable_diffusion.inverse_stable_diffusion import InversableStableDiffusionPipeline
from src.utils.optim_utils import *
from src.utils.io_utils import *
import statistics

In [2]:
def nmae(tensor1, tensor2):
    assert tensor1.shape == tensor2.shape, "两个张量的形状必须相同"
    mae = torch.mean(torch.abs(tensor1 - tensor2))
    nmae = mae / torch.mean(torch.abs(tensor1))

    return nmae.item()

In [3]:
guidance_scale=3
model_id= 'C:\\Users\\18245\\.cache\\huggingface\\hub\\models--stabilityai--stable-diffusion-2-1-base\\snapshots\\5ede9e4bf3e3fd1cb0ef2f7a3fff13ee514fdf06'
num_inference_steps=50
inv_order=1


scheduler = DPMSolverMultistepScheduler(
        beta_end=0.012,
        beta_schedule='scaled_linear',
        beta_start=0.00085,
        num_train_timesteps=1000,
        prediction_type="epsilon",
        steps_offset=1, 
        trained_betas=None,
        solver_order=inv_order,
    )

pipe = InversableStableDiffusionPipeline.from_pretrained(
        model_id,
        scheduler=scheduler,
        torch_dtype=torch.float16,
        ).to("cuda")

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

In [4]:

prompt="a fat yellow cat"
text_embeddings_tuple = pipe.encode_prompt(
                prompt, 'cuda', 1, guidance_scale > 1.0, None)
text_embeddings = torch.cat([text_embeddings_tuple[1], text_embeddings_tuple[0]])

# generate init latent
set_random_seed(100)
init_latents = pipe.get_random_latents()

z0 = pipe(
            prompt,
            guidance_scale=guidance_scale,
            num_inference_steps=num_inference_steps,
            latents=init_latents,
            )

reversed_latents = pipe.forward_diffusion(
            latents=z0, 
            text_embeddings=text_embeddings,
            guidance_scale=guidance_scale,
            num_inference_steps=num_inference_steps,
            inv_order=inv_order,
        )

err_T0T=(((init_latents-reversed_latents).norm()/init_latents.norm()).item())**2
print(f"T0T NMSE: {err_T0T}")
print("MAE:",nmae(reversed_latents,init_latents))
print("l2:",torch.norm(reversed_latents-init_latents).item())


  0%|          | 0/50 [00:00<?, ?it/s]

C:\Users\18245\.conda\envs\torch23\Lib\site-packages\diffusers\models\attention_processor.py:1036: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  hidden_states = F.scaled_dot_product_attention(


  0%|          | 0/50 [00:00<?, ?it/s]

T0T NMSE: 7.21058459021151e-05
MAE: 0.00740814208984375
l2: 1.0966796875
